# Notebook for quickly pulling data

In [ ]:
import pandas as pd

import processing.acled_events_processing as acled
import processing.acled_text_processing as notes
import processing.food_prices_processing as food
import processing.rainfall_processing as rain
from utils.dates import END_DATE, TRAIN_START_DATE


In [ ]:
processed_acled_df, acled_predictor_cols, raw_acled_df = acled.get_clean_data(
        k=1.75, event_col="event_type"
    )

all_regions = processed_acled_df["region"].unique()
all_months = pd.period_range(TRAIN_START_DATE, END_DATE, freq="M")

In [ ]:
processed_rain_df, rain_predictor_cols = rain.get_clean_data(
                all_regions=all_regions,
                all_months=all_months,
            )

In [ ]:
processed_food_df, food_predictor_cols = food.get_clean_data(
                all_regions=all_regions,
                all_months=all_months,
            )

In [ ]:
processed_notes_df, notes_prediction_cols = notes.get_clean_data(
                df=raw_acled_df,
                all_regions=all_regions,
                all_months=all_months,
                conflict_only=True,
                # Reads locally saved embeddings from unless FORCE_DOWNLOAD is set
            )

In [ ]:
from transformers import AutoModel, AutoTokenizer

import processing.acled_events_processing as acled
from processing.acled_text_processing import (
    check_max_tokens,
    get_monthly_regional_embeddings,
    remove_dates,
)

In [ ]:
model_name = "eventdata-utd/ConfliBERT-scr-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()  # Tells model I am not training

sample = raw_acled_df.head(10)
df_regex = remove_dates(sample)
check_max_tokens(tokenizer, df_regex)
df_monthly = get_monthly_regional_embeddings(df_regex, tokenizer, model)

df_monthly.to_pickle("data/acled/acled_monthly_regional_embeddings.pkl")